# Practical 8: YOLO Object Detection on the Fruits Dataset

**Aim:** To train a YOLOv8 object detector (yolov8n, COCO-pretrained) that locates and classifies fruits in images with bounding boxes, after fixing the dataset's incorrect label files.

> Reconstructed code (the source PDF has no code). Not executed by the author - run cells in order (Kernel -> Restart & Run All) and check dataset paths in the first cells. Expected values: mAP@0.5 ~0.85-0.95; precision and recall ~0.90+.; TRUE object detection (boxes + classes). Use yolov8n.pt (COCO-pretrained).

## Cell 1 - Install and import

**What it does:**
- `re` is used to read class names from file names; `shutil`/`random` for copying and splitting files.

*Expected output (illustrative):* No output (after installation).

In [ ]:
!pip install -q ultralytics

import re
import random
import shutil
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

IMG_EXT = {".jpg", ".jpeg", ".png"}

## Cell 2 - Inspect the dataset and expose the label bug

**What it does:**
- `SRC` is the dataset folder (change if yours has another name). `rglob` searches all sub-folders.
- `labels` maps each label file name (without extension) to its path.
- Printing the set of class ids shows the bug: only `{'0'}` appears, although there are 10 fruit types.

*Expected output (illustrative):* Images: 400  Label files: 400
Class ids found in label files: {'0'}
Example image names: ['01_Banana.jpg', ...]

In [ ]:
SRC = Path("fruits")

images = sorted(p for p in SRC.rglob("*") if p.suffix.lower() in IMG_EXT)
labels = {p.stem: p for p in SRC.rglob("*.txt")}
print("Images:", len(images), " Label files:", len(labels))

ids_found = set()
for p in labels.values():
    for line in p.read_text().splitlines():
        if line.strip():
            ids_found.add(line.split()[0])
print("Class ids found in label files:", ids_found)

print("Example image names:", [p.name for p in images[:5]])

## Cell 3 - Get the true class from each file name

**What it does:**
- `01_Banana.jpg` -> stem `01_Banana` -> text after the first underscore `Banana` -> lower-case `banana`. A trailing number (e.g. `Banana_2`) is removed by the regular expression.
- `class_names` is the sorted list of distinct fruit names; `class_to_id` assigns each a number 0..9. **Check that exactly 10 classes appear.**

*Expected output (illustrative):* Number of classes: 10 followed by a dictionary such as {'apple': 0, 'banana': 1, ...} (your fruit names).

In [ ]:
def class_from_name(path):
    stem = path.stem
    name = stem.split("_", 1)[1] if "_" in stem else stem
    name = re.sub(r"[_\-\s]*\d+$", "", name)
    return name.strip().lower()


class_names = sorted({class_from_name(p) for p in images})
class_to_id = {name: i for i, name in enumerate(class_names)}

print("Number of classes:", len(class_names))
print(class_to_id)

## Cell 4 - Build the YOLO dataset with corrected labels (stratified 80/20)

**What it does:**
- Creates the folders YOLO expects: `images/train`, `images/val`, `labels/train`, `labels/val`.
- Images are split **per class** (20% of each class to validation) so every fruit appears in both sets.
- For every image the label file is rewritten: the wrong first number (always 0) is replaced by the correct `class_id`; the four box numbers are kept unchanged.
- `missing_labels` should be 0.

*Expected output (illustrative):* Train images: 320
Val images  : 80
Images without label file: 0

In [ ]:
random.seed(42)
OUT = Path("fruits_yolo")

for split in ("train", "val"):
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

by_class = {}
for p in images:
    by_class.setdefault(class_from_name(p), []).append(p)

split_of = {}
for name, files in by_class.items():
    random.shuffle(files)
    n_val = max(1, int(round(0.2 * len(files))))
    for i, p in enumerate(files):
        split_of[p] = "val" if i < n_val else "train"

missing_labels = 0
for p, split in split_of.items():
    class_id = class_to_id[class_from_name(p)]
    shutil.copy(p, OUT / "images" / split / p.name)

    new_lines = []
    src_label = labels.get(p.stem)
    if src_label is None:
        missing_labels += 1
    else:
        for line in src_label.read_text().splitlines():
            parts = line.split()
            if len(parts) == 5:
                new_lines.append(" ".join([str(class_id)] + parts[1:]))
    (OUT / "labels" / split / (p.stem + ".txt")).write_text("\n".join(new_lines))

print("Train images:", sum(1 for s in split_of.values() if s == "train"))
print("Val images  :", sum(1 for s in split_of.values() if s == "val"))
print("Images without label file:", missing_labels)

## Cell 5 - Write the dataset YAML file

**What it does:**
- `data.yaml` tells YOLO where the images are and the names of the classes. YOLO finds the label folders automatically by replacing `images` with `labels` in the path.

*Expected output (illustrative):* Prints the YAML: path, train, val and the 10 numbered class names.

In [ ]:
yaml_lines = [
    "path: " + OUT.resolve().as_posix(),
    "train: images/train",
    "val: images/val",
    "names:",
]
for i, name in enumerate(class_names):
    yaml_lines.append("  " + str(i) + ": " + name)

(OUT / "data.yaml").write_text("\n".join(yaml_lines) + "\n")
print((OUT / "data.yaml").read_text())

## Cell 6 - Train YOLOv8n (transfer learning from COCO)

**What it does:**
- `yolov8n.pt` = nano detection model pre-trained on COCO (downloaded automatically).
- `imgsz=640` is YOLO's default input size; `batch=16` is safe for most GPUs (if you get CUDA out-of-memory, reduce it - see troubleshooting). `epochs=50` is a reconstructed value.
- YOLO's training uses mosaic augmentation, HSV colour jitter, flips and scale/translate automatically.
- Per epoch it prints box loss, class loss, DFL loss and validation precision, recall, mAP50, mAP50-95.

*Expected output (illustrative):* A progress table per epoch. By the end: precision/recall about 0.9+, mAP@0.5 about 0.85-0.95 (from your PDF).

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str((OUT / "data.yaml").resolve()),
    epochs=50,
    imgsz=640,
    batch=16,
)

save_dir = Path(model.trainer.save_dir)
print("Results saved in:", save_dir)

## Cell 7 - Training curves (loss, precision, recall, mAP)

**What it does:**
- `results.csv` is written by YOLO after every epoch. Losses should decrease while precision, recall and mAP increase.
- Ultralytics also saves ready-made graphs (`results.png`, `confusion_matrix.png`, PR/F1 curves) in `save_dir`.

*Expected output (illustrative):* Three plots: falling losses, rising precision/recall/mAP.

In [ ]:
df = pd.read_csv(save_dir / "results.csv")
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df["epoch"], df["train/box_loss"], label="Train box loss")
axes[0].plot(df["epoch"], df["val/box_loss"], label="Val box loss")
axes[0].set_title("Box loss")
axes[0].legend()

axes[1].plot(df["epoch"], df["train/cls_loss"], label="Train class loss")
axes[1].plot(df["epoch"], df["val/cls_loss"], label="Val class loss")
axes[1].set_title("Class loss")
axes[1].legend()

axes[2].plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
axes[2].plot(df["epoch"], df["metrics/recall(B)"], label="Recall")
axes[2].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5")
axes[2].set_title("Validation metrics")
axes[2].legend()

for ax in axes:
    ax.set_xlabel("Epoch")
plt.tight_layout()
plt.show()

## Cell 8 - Evaluate: precision, recall, mAP and per-class AP

**What it does:**
- Loads the best weights and runs validation on the 80 validation images.
- `mp` / `mr` = mean precision / recall over classes; `map50` = mAP at IoU 0.5; `map` = mAP averaged over IoU 0.5-0.95.
- The per-class list reveals the weakest fruits (expected: hog plum, mango, guava).

*Expected output (illustrative):* mAP@0.5 about 0.85-0.95, precision and recall about 0.90+ (from your PDF); per-class numbers vary.

In [ ]:
best = YOLO(str(save_dir / "weights" / "best.pt"))
metrics = best.val(data=str((OUT / "data.yaml").resolve()))

print("Precision   :", round(float(metrics.box.mp), 4))
print("Recall      :", round(float(metrics.box.mr), 4))
print("mAP@0.5     :", round(float(metrics.box.map50), 4))
print("mAP@0.5:0.95:", round(float(metrics.box.map), 4))

print("\nPer-class mAP@0.5:0.95")
for class_index, ap in zip(metrics.box.ap_class_index, metrics.box.maps[metrics.box.ap_class_index]):
    print(" ", best.names[int(class_index)], round(float(ap), 3))

## Cell 9 - Confusion matrix and detections on validation images

**What it does:**
- The confusion matrix was saved by YOLO during validation; here it is displayed.
- `best.predict` runs detection with a confidence threshold of 0.25 (boxes below it are dropped). NMS is applied automatically.
- `r.plot()` returns the image with boxes and labels drawn in BGR colour order; `[:, :, ::-1]` converts it to RGB for Matplotlib.

*Expected output (illustrative):* A confusion matrix image and 6 fruit photos with coloured boxes labelled 'banana 0.93' etc.; console lines listing predicted class and confidence.

In [ ]:
cm_image = Image.open(save_dir / "confusion_matrix.png")
plt.figure(figsize=(8, 8))
plt.imshow(cm_image)
plt.axis("off")
plt.title("Confusion matrix (from YOLO validation)")
plt.show()

val_images = sorted((OUT / "images" / "val").glob("*"))[:6]
preds = best.predict(source=[str(p) for p in val_images], conf=0.25, verbose=False)

plt.figure(figsize=(14, 8))
for i, r in enumerate(preds):
    plt.subplot(2, 3, i + 1)
    plt.imshow(r.plot()[:, :, ::-1])
    plt.axis("off")
    for box in r.boxes:
        print(val_images[i].name, "->", best.names[int(box.cls)], round(float(box.conf), 2))
plt.tight_layout()
plt.show()

## Result / Conclusion

After correcting the faulty label files, a COCO-pretrained YOLOv8n detector trained on 320 fruit images located and classified fruits with mAP@0.5 around 0.85-0.95.